# 🎙️ Chakravyuh: Raghunath Voice Model Overnight Fine-Tuning
**Target:** Next Gen Hackathon 2026 Bengaluru  
**Speaker:** Joel Anthony (`uncoalesced`) — Raghunath Elderly Persona  
**Base Voice:** `hi_IN-rohan-medium` (Hindi / Hinglish VITS)  
**Runtime:** GPU (T4 or higher on Google Colab / Kaggle)  
**Estimated Duration:** ~1 to 1.5 hours (100 epochs on ~4 minutes of audio)

### Step 1: Check GPU Runtime
Ensure you have selected **Runtime > Change runtime type > T4 GPU** in Google Colab.

In [ ]:
!nvidia-smi

### Step 2: Install System Dependencies & Piper Training Suite

In [ ]:
!apt-get update -qq
!apt-get install -y -qq espeak-ng libespeak-ng-dev build-essential git wget

# piper-phonemize 1.1.0 only ships cp39/cp310 wheels and piper_train needs torch<2,
# so Colab's Python 3.11+ can't install them. Use an isolated Python 3.10 venv.
!pip install -q uv
!uv venv --clear --python 3.10 /content/piper_env
PY = "/content/piper_env/bin/python"
!uv pip install --python {PY} "torch==1.13.1" --index-url https://download.pytorch.org/whl/cu117
!uv pip install --python {PY} cython "numpy<2" pip setuptools wheel

# Clone Piper repo and install training package from src/python
%cd /content
!rm -rf piper
!git clone https://github.com/rhasspy/piper.git
%cd /content/piper/src/python
!uv pip install --python {PY} -r requirements.txt
!PATH=/content/piper_env/bin:$PATH bash build_monotonic_align.sh
!uv pip install --python {PY} -e .
# pytorch-lightning 1.7 needs pkg_resources (removed in setuptools>=70) and the old torchmetrics API
!uv pip install --python {PY} "setuptools<70" "torchmetrics<1" six "protobuf<4" "tensorboard==2.11.2"

# Fail fast if the environment is broken
!{PY} -c "import piper_phonemize, pytorch_lightning, torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

### Step 3: Download Base Pre-trained Checkpoint (`hi_IN-rohan-medium`)

In [ ]:
!mkdir -p /content/checkpoints
!wget -c -O /content/checkpoints/base_rohan.ckpt \
  "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/hi/hi_IN/rohan/medium/epoch=3190-step=309852.ckpt"
!ls -lh /content/checkpoints/base_rohan.ckpt

### Step 4: Upload & Extract Dataset (`raghunath_dataset.zip`)
Upload `raghunath_dataset.zip` from your local folder:  
`C:\Users\Joel Anthony\Documents\Phile\Heafod\Heafod\Chakravyuh\voice_studio\data\raghunath_dataset.zip`

In [ ]:
import os
from google.colab import files

%cd /content
if not os.path.exists("/content/raghunath_dataset.zip"):
    print("Please upload raghunath_dataset.zip now:")
    uploaded = files.upload()

!rm -rf /content/dataset
!mkdir -p /content/dataset
!unzip -q /content/raghunath_dataset.zip -d /content/dataset

print("\nDataset contents:")
!ls -lh /content/dataset
!head -n 5 /content/dataset/metadata.csv

### Step 5: Preprocess Dataset with Piper Phonemizer (Devanagari / Hindi)

In [ ]:
%cd /content/piper/src/python
!rm -rf /content/dataset_preprocessed
!/content/piper_env/bin/python -m piper_train.preprocess \
  --language hi \
  --input-dir /content/dataset \
  --output-dir /content/dataset_preprocessed \
  --dataset-format ljspeech \
  --sample-rate 22050

print("\nPreprocessed files:")
!ls -lh /content/dataset_preprocessed

### Step 6: Overnight Fine-Tuning (100 Epochs: base ckpt is epoch 3190, so max_epochs=3290)
This will train on the T4 GPU. On 41 clips (~4 min audio), 100 epochs will take **~1 to 1.5 hours**.

In [ ]:
%cd /content/piper/src/python

!/content/piper_env/bin/python -m piper_train \
  --dataset-dir /content/dataset_preprocessed \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --max_epochs 3290 \
  --resume_from_checkpoint /content/checkpoints/base_rohan.ckpt \
  --checkpoint-epochs 1 \
  --precision 16

# Export + download immediately, so a runtime reset can't lose the result
import glob, os
from google.colab import files
best_ckpt = sorted(glob.glob("/content/dataset_preprocessed/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)[-1]
!/content/piper_env/bin/python -m piper_train.export_onnx "{best_ckpt}" /content/raghunath_elderly-medium.onnx
!cp /content/dataset_preprocessed/config.json /content/raghunath_elderly-medium.onnx.json
!cd /content && zip -j raghunath_model_exported.zip raghunath_elderly-medium.onnx raghunath_elderly-medium.onnx.json
files.download("/content/raghunath_model_exported.zip")

### Step 7: Export to ONNX (`raghunath_elderly-medium.onnx`)

In [ ]:
import glob, os

# Search all potential directories for fine-tuned checkpoints
checkpoints = sorted(
    [p for p in glob.glob("/content/**/*.ckpt", recursive=True) if "base_rohan" not in p],
    key=os.path.getmtime
)

if not checkpoints:
    print("❌ No fine-tuned checkpoints found yet!")
    print("Please check Step 6: did the training command finish successfully?")
    raise FileNotFoundError("No training checkpoints found. Check Step 6 output for errors or wait for training to finish.")

best_ckpt = checkpoints[-1]
print(f"✅ Selected latest checkpoint: {best_ckpt}")

%cd /content/piper/src/python
!/content/piper_env/bin/python -m piper_train.export_onnx "{best_ckpt}" /content/raghunath_elderly-medium.onnx

# Copy preprocessed JSON configuration
!cp /content/dataset_preprocessed/config.json /content/raghunath_elderly-medium.onnx.json

!ls -lh /content/raghunath_elderly-medium.onnx*
print("\nONNX export completed!")

### Step 8: Package & Download Trained Model
Download the zip and place the `.onnx` and `.onnx.json` into `E:\Chakravyuh\models\piper\`.

In [ ]:
%cd /content
!zip -j raghunath_model_exported.zip /content/raghunath_elderly-medium.onnx /content/raghunath_elderly-medium.onnx.json

from google.colab import files
files.download("/content/raghunath_model_exported.zip")